# IMUSA V2 — Multi-Account Colab Worker 1 (Folds 0 & 1)
This notebook runs **Stratified 5-Fold Cross-Validation Folds 0 & 1** on Account 1.
Saves fold checkpoints and out-of-fold probability files to `outputs/v2/` and Google Drive.

In [ ]:
# 1. Environment & GPU Setup
!nvidia-smi
!pip install -q uv
import os
import sys

if not os.path.exists("imusa-multimodal-sentiment"):
    !git clone https://github.com/shubhojit-mitra-dev/imusa-multimodal-sentiment.git
%cd /content/imusa-multimodal-sentiment
!git pull origin main
!pip install -e libs/imusa
sys.path.insert(0, "/content/imusa-multimodal-sentiment/libs/imusa/src")

In [ ]:
# 2. Google Drive Integration & Automatic data.zip Handling
import os
import shutil

from google.colab import drive, files

drive.mount("/content/drive", force_remount=False)
gdrive_zip = "/content/drive/MyDrive/data.zip"
local_zip = "/content/imusa-multimodal-sentiment/data.zip"

if os.path.exists(gdrive_zip):
    print("Found data.zip in Google Drive. Copying locally...")
    shutil.copy(gdrive_zip, local_zip)
elif not os.path.exists(local_zip):
    print("data.zip not found in Google Drive (MyDrive/data.zip).")
    print("Please select and upload data.zip from your computer now:")
    uploaded = files.upload()
    for fname in uploaded.keys():
        if fname.endswith(".zip"):
            shutil.move(fname, local_zip)
            break

# Copy to Google Drive for future runs
if os.path.exists(local_zip) and not os.path.exists(gdrive_zip):
    print("Saving data.zip to Google Drive (MyDrive/data.zip) for future runs...")
    try:
        shutil.copy(local_zip, gdrive_zip)
        print("Saved to Google Drive.")
    except Exception as e:
        print(f"Note: Could not copy to Drive: {e}")

# Unzip dataset
!unzip -q -o /content/imusa-multimodal-sentiment/data.zip -d /content/imusa-multimodal-sentiment/
print("Dataset extracted to data/.")

In [ ]:
# 3. Run Fold 0 Training (LP-FT, MuRIL, Label Smoothing, Mixup)
!python scripts/train_kfold.py --fold 0 --num-folds 5 --epochs 10 --lp-epochs 3

In [ ]:
# 4. Run Fold 1 Training
!python scripts/train_kfold.py --fold 1 --num-folds 5 --epochs 10 --lp-epochs 3

In [ ]:
# 5. Compress Fold 0 & 1 Checkpoints & OOF outputs, and save to Google Drive
!zip -r fold_0_1_outputs.zip outputs/v2/checkpoints/best_model_fold_0.pt outputs/v2/checkpoints/best_model_fold_1.pt outputs/v2/oof_probs_fold_*.npy outputs/v2/oof_targets_fold_*.npy
if os.path.exists("/content/drive/MyDrive"):
    shutil.copy("fold_0_1_outputs.zip", "/content/drive/MyDrive/fold_0_1_outputs.zip")
    print("Saved fold_0_1_outputs.zip to Google Drive (MyDrive/fold_0_1_outputs.zip).")